In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostRegressor

In [2]:
data_train = pd.read_csv("./used-cars-price-prediction-19ds/train.csv")
data_test = pd.read_csv("./used-cars-price-prediction-19ds/test.csv")

In [3]:
RANDOM_STATE = 12345
TRIM_LEVELS = {
    "ce": ["classic edition", "custom edition", "ce"], 
    "dx": ["deluxe", "dx"],
    "dl": ["deluxe level", "dl"],
    "ex": ["extra", "ex"], 
    "gl": ["grade level", "gl"],
    "gle": ["grade level extra", "gle"],
    "gt": ["grand touring", "gt"],
    "lx": ["luxury", "lx"],
    "le": ["luxury edition", "le"],
    "ls": ["luxury sport", "luxury special", "ls"],
    "lt": ["luxury touring", "lt"],
    "ltd": ["limited", "ltd"],
    "ltz": ["luxury touring special", "ltz"],
    "se": ["sport edition", "special edition", "special equipment", "se"],
    "sl": ["standart level", "standart", "sl"],
    "sle": ["standard level extra", "sle"],
    "slt": ["standard level touring", "slt"],
    "sv": ["Special Version", "sv"],
    "xlt": ["extra level touring", "xlt"],
    "base": ["base"],
    "e": ["edition", "e"], 
    "t": ["touring", "t"],
}

In [4]:
def make_unique(data):
    if "ford" in data:
        return "ford"
    if "gmc" in data:
        return "gmc"
    if "land" in data:
        return "landrover"
    if "mercedes" in data:
        return "mercedes"
    if data == "vw":
        return "volkswagen"
    if "dodge" in data:
        return "dodge"
    if "mazda" in data:
        return "mazda"
    else:
        return data

In [5]:
def body_unique(data):
    if data.find("cab") != -1 or data == "koup":
        return "pick-up"
    if data.find("convertible") != -1:
        return "convertible"
    if data.find("coupe") != -1:
        return "coupe"
    if data.find("wagon") != -1:
        return "wagon"
    if data.find("van") != -1:
        return "van"
    if data.find("sedan") != -1:
        return "sedan"
    if data.find("van") != -1:
        return "van"
    else:
        return data

In [6]:
def classify_trim_levels(trim_levels, data):
    for key in trim_levels:
        for index in trim_levels[key]:
            if index in data:
                return key
    return "unknown"

In [7]:
def trim_unique(data):
    return classify_trim_levels(TRIM_LEVELS, data)

In [8]:
def preprocessing(data):
    data = data.drop(["vin", "seller", "saledate"], axis=1)

    data["make"] = data["make"].apply(lambda x: str(x).lower())
    data["model"] = data["model"].apply(lambda x: str(x).lower())
    data["trim"] = data["trim"].apply(lambda x: str(x).lower())
    data["body"] = data["body"].apply(lambda x: str(x).lower())
    data["transmission"] = data["transmission"].apply(lambda x: str(x).lower())
    data["state"] = data["state"].apply(lambda x: str(x).lower())
    data["color"] = data["color"].apply(lambda x: str(x).lower())
    data["interior"] = data["interior"].apply(lambda x: str(x).lower())

    data["year"] = data["year"].astype(int)
    
    data["trim"] = data["trim"].apply(trim_unique)
    data["make"] = data["make"].apply(make_unique)
    data["body"] = data["body"].apply(body_unique)

    mean_value_odometer = data["odometer"].mean()
    mean_values_odometer = list(data.groupby("condition")["odometer"].mean().values)
    mean_values_odometer_class = list(data.groupby("condition")["odometer"].mean().index)
    index_odometer = mean_values_odometer.index(max(filter(lambda x: x < mean_value_odometer, mean_values_odometer), default=None))
    class_value = mean_values_odometer_class[index_odometer]
    data[(data["odometer"] > 0.19 * (10**6))]["odometer"] = float(mean_value_odometer)
    data[data["odometer"] == mean_value_odometer]["condition"] = float(class_value)

    return data

In [9]:
def getfullitemsforOHE(wholedf,featlist,sort=True):
    def sortornot(X):
        if sort==False:
            return X
        else:
            return sorted(X)
       
    fulllist=[]
    for feat in featlist:
        fulllist.append(sortornot(wholedf[feat].unique()))
    return fulllist

In [10]:
data_train = preprocessing(data_train)
data_test = preprocessing(data_test)

C:\Users\Rog\AppData\Local\Temp\ipykernel_14488\1730885387.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[(data["odometer"] > 0.19 * (10**6))]["odometer"] = float(mean_value_odometer)
C:\Users\Rog\AppData\Local\Temp\ipykernel_14488\1730885387.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[(data["odometer"] > 0.19 * (10**6))]["odometer"] = float(mean_value_odometer)


In [12]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 430811 entries, 0 to 440235
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          430811 non-null  int32  
 1   make          430811 non-null  object 
 2   model         430811 non-null  object 
 3   trim          430811 non-null  object 
 4   body          430811 non-null  object 
 5   transmission  430811 non-null  object 
 6   state         430811 non-null  object 
 7   condition     430811 non-null  float64
 8   odometer      430811 non-null  float64
 9   color         430811 non-null  object 
 10  interior      430811 non-null  object 
 11  sellingprice  430811 non-null  int64  
dtypes: float64(2), int32(1), int64(1), object(8)
memory usage: 41.1+ MB


In [13]:
full_data = pd.concat([data_train, data_test])
full_data = full_data.reset_index(drop=True)

In [14]:
cat_columns = ["make", "model", "trim", "body", "transmission", "state", "color", "interior"]

In [15]:
cats = getfullitemsforOHE(full_data, cat_columns)

In [16]:
ohe=OneHotEncoder(categories=cats, sparse=False, handle_unknown="ignore")

X_train = ohe.fit_transform(data_train[cat_columns])
ohe_train = pd.DataFrame(X_train,columns=ohe.get_feature_names_out(cat_columns))

X_test = ohe.fit_transform(data_test[cat_columns])
ohe_test = pd.DataFrame(X_test,columns=ohe.get_feature_names_out(cat_columns))

c:\Users\Rog\Desktop\Programing\Python\envs\Kaggle\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\Rog\Desktop\Programing\Python\envs\Kaggle\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [17]:
data_train = data_train.drop(cat_columns, axis=1)
data_train = data_train.join(ohe_train)

data_test = data_test.drop(cat_columns, axis=1)
data_test = data_test.join(ohe_test)

In [18]:
features = data_train.drop(["sellingprice"], axis=1)
target = data_train["sellingprice"]

In [19]:
cat = CatBoostRegressor(n_estimators=500, depth=16, random_state=RANDOM_STATE, silent=True, task_type="GPU", devices="0:1")

In [20]:
cat.fit(features, target, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

KeyboardInterrupt: 

In [ ]:
results = pd.DataFrame(pd.read_csv("./used-cars-price-prediction-19ds/test.csv")["vin"]).join(pd.DataFrame(cat.predict(data_train)))

In [ ]:
display(results)

In [ ]:
results.columns = ["vin", "sellingprice"]

In [ ]:
results.to_csv("results.csv", index=False)